In [ ]:
# Plot a 9-day agile tariff window centered on the selected 7-day period in FullEnergyOptimizationDemo11.
# Add a second subplot with the first two selected days after fixed non-circular offsets: day1 +2h, day2 -1.5h.
import json
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

# Force a white plotting style regardless of the notebook's global theme.
plt.style.use('default')
mpl.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'savefig.facecolor': 'white',
    'text.color': 'black',
    'axes.labelcolor': 'black',
    'axes.edgecolor': 'black',
    'axes.titlecolor': 'black',
    'xtick.color': 'black',
    'ytick.color': 'black',
    'grid.color': '#D0D0D0'
})

# Resolve repository root from current notebook location.
repo_root = Path.cwd().resolve()
if not (repo_root / '.git').exists():
    for parent in repo_root.parents:
        if (parent / '.git').exists():
            repo_root = parent
            break

demo_nb_path = repo_root / 'Codes' / 'FullEnergyOptimizationDemo11.ipynb'
if not demo_nb_path.exists():
    raise FileNotFoundError(f'Cannot find notebook: {demo_nb_path}')

nb = json.loads(demo_nb_path.read_text(encoding='utf-8'))
workflow_src = None
for cell in nb.get('cells', []):
    if cell.get('cell_type') != 'code':
        continue
    src = ''.join(cell.get('source', []))
    if 'workflow_cfg' in src:
        workflow_src = src
        break

if workflow_src is None:
    raise ValueError('workflow_cfg was not found in FullEnergyOptimizationDemo11.ipynb')

def _match(pattern: str, text: str, cast=str, default=None):
    m = re.search(pattern, text)
    if not m:
        return default
    return cast(m.group(1))

step = _match(r"['\"]step['\"]\s*:\s*['\"]([^'\"]+)['\"]", workflow_src, str, '30min')
start_date_str = _match(r"['\"]start_date['\"]\s*:\s*pd\.Timestamp\(\s*['\"]([^'\"]+)['\"]\s*\)", workflow_src)
if start_date_str is None:
    start_date_str = _match(r"['\"]start_date['\"]\s*:\s*['\"]([^'\"]+)['\"]", workflow_src)
selected_days = _match(r"['\"]n_days['\"]\s*:\s*([0-9]+)", workflow_src, int, 7)

if start_date_str is None:
    raise ValueError('Could not parse workflow_cfg start_date from FullEnergyOptimizationDemo11.ipynb')
if int(selected_days) < 2:
    raise ValueError('Need at least 2 selected days to plot day1/day2 offsets.')

selected_start = pd.Timestamp(start_date_str)
selected_end = selected_start + pd.Timedelta(days=int(selected_days))

# Build a 9-day agile tariff with one offset day on each side.
source_dir = repo_root / 'Codes' / 'sourcecode'
source_dir_str = str(source_dir)
if source_dir_str in sys.path:
    sys.path.remove(source_dir_str)
sys.path.insert(0, source_dir_str)

import stochastic_baseload_multiple_building_simulation_and_aggregation as sbm

tariff_start = selected_start - pd.Timedelta(days=1)
tariff_days = int(selected_days) + 2
tariff_9d = sbm.build_tariff(tariff_start, n_days=tariff_days, step=step, type='agile')

# Build first two selected days and apply fixed non-circular offsets.
step_td = pd.Timedelta(step)
hours_per_step = float(step_td / pd.Timedelta(hours=1))
steps_per_day = int(round(pd.Timedelta(days=1) / step_td))
offset_hours = [2.0, -1.5]
offset_steps = [int(round(h / hours_per_step)) for h in offset_hours]

two_day_start = selected_start
two_day_end = selected_start + pd.Timedelta(days=2)
tariff_2d = tariff_9d.loc[(tariff_9d.index >= two_day_start) & (tariff_9d.index < two_day_end), ['elec_price']].copy()
if tariff_2d.empty:
    raise ValueError('Two-day tariff window is empty. Check start date and tariff generation.')

original_vals = tariff_2d['elec_price'].to_numpy(copy=True)
full_prices = tariff_9d['elec_price'].to_numpy(copy=True)
offset_vals = np.full(len(original_vals), np.nan, dtype=float)
for day_idx, offset_h in enumerate(offset_hours):
    day_start = day_idx * steps_per_day
    day_end = min(day_start + steps_per_day, len(original_vals))
    if day_end <= day_start:
        continue
    target_idx = tariff_2d.index[day_start:day_end]
    source_idx = target_idx - pd.Timedelta(hours=offset_h)
    source_pos = tariff_9d.index.get_indexer(source_idx)
    if np.any(source_pos < 0):
        raise ValueError('Offset source index is outside 9-day window. Increase surrounding tariff days.')
    offset_vals[day_start:day_end] = full_prices[source_pos]
tariff_2d['elec_price_offset'] = offset_vals

day1_start = selected_start
day1_end = selected_start + pd.Timedelta(days=1)
day2_start = day1_end
day2_end = day2_start + pd.Timedelta(days=1)
day1_source_start = day1_start - pd.Timedelta(hours=offset_hours[0])
day1_source_end = day1_end - pd.Timedelta(hours=offset_hours[0])
day2_source_start = day2_start - pd.Timedelta(hours=offset_hours[1])
day2_source_end = day2_end - pd.Timedelta(hours=offset_hours[1])

fig, axes = plt.subplots(2, 1, figsize=(19, 9.2), facecolor='white')
ax0, ax1 = axes
for ax in (ax0, ax1):
    ax.set_facecolor('white')
    ax.tick_params(axis='both', colors='black')
    for spine in ax.spines.values():
        spine.set_color('black')

ax0.step(tariff_9d.index, tariff_9d['elec_price'], where='post', linewidth=2.0, color='tab:blue', label='Agile electricity price')
if 'gas_price' in tariff_9d.columns:
    ax0.step(tariff_9d.index, tariff_9d['gas_price'], where='post', linewidth=1.2, linestyle='--', color='tab:orange', label='Gas price')
ax0.axvspan(tariff_start, selected_start, color='#E0E0E0', alpha=0.55, label='Outside selected 7 days')
ax0.axvspan(selected_end, selected_end + pd.Timedelta(days=1), color='#E0E0E0', alpha=0.55)
ax0.axvline(selected_start, color='black', linestyle=':', linewidth=1.2)
ax0.axvline(selected_end, color='black', linestyle=':', linewidth=1.2)
ax0.axvspan(day1_source_start, day1_source_end, color='#E57373', alpha=0.28, label='24h source window for day1 (+2h roll)')
ax0.axvspan(day2_source_start, day2_source_end, color='#9575CD', alpha=0.28, label='24h source window for day2 (-1.5h roll)')
ax0.axvline(day1_source_start, color='tab:red', linestyle='--', linewidth=1.4)
ax0.axvline(day1_source_end, color='tab:red', linestyle='--', linewidth=1.4)
ax0.axvline(day2_source_start, color='tab:purple', linestyle='--', linewidth=1.4)
ax0.axvline(day2_source_end, color='tab:purple', linestyle='--', linewidth=1.4)
ax0.set_title(f'9-day original agile tariff centered on selected {selected_days}-day period')
ax0.set_ylabel('Price')
ax0.set_xlabel('Time')
ax0.grid(True, alpha=0.65)
ax0.legend(loc='upper right', facecolor='white', edgecolor='black', framealpha=1.0)

ax1.step(tariff_2d.index, tariff_2d['elec_price'], where='post', linewidth=2.0, color='tab:blue', label='Original (2 days)')
ax1.step(tariff_2d.index, tariff_2d['elec_price_offset'], where='post', linewidth=2.0, color='tab:red', label='Offset (non-circular): day1 +2h, day2 -1.5h')
ax1.axvline(selected_start + pd.Timedelta(days=1), color='black', linestyle=':', linewidth=1.0, label='Day boundary')
ax1.set_title('First 2 selected days after non-circular offsets')
ax1.set_ylabel('Electricity price')
ax1.set_xlabel('Time')
ax1.grid(True, alpha=0.65)
ax1.legend(loc='upper right', facecolor='white', edgecolor='black', framealpha=1.0)

plt.tight_layout()
plt.show()

print(f'Selected window: {selected_start} to {selected_end}')
print(f'Plotted tariff window: {tariff_start} to {selected_end + pd.Timedelta(days=1)}')
print(f'Offset hours (day1, day2): {offset_hours}')
print(f'Offset steps (day1, day2): {offset_steps}')
